# FlashNystrom — Colab experiments → paper artifacts (saved to Drive)

Run top to bottom on an A100/L4: clone → build → verify → experiments → aggregate → figures → zip. Everything (JSON + logs + PDF/PNG figures + zip) is written to Google Drive, so a disconnect loses nothing.

Two FlashNystrom pseudoinverse paths are compared throughout:
- **`flash_nystrom`** — the faithful **scalar fp32** Newton-Schulz pinv (the default). Matches the pure-PyTorch reference to ~3e-4 and is ~3x faster than it.
- **`flash_nystrom_tc`** — the opt-in **tf32 tensor-core** pinv. ~4x faster than the reference but carries an N-independent ~1-3% operator error.

The ridge (`kappa_star`) is **threaded identically to FlashNystrom and the reference**, so every comparison is matched. A separate **`nystrom_vanilla`** arm (the reference forced to kappa=0, i.e. the original Nystromformer with no ridge) runs alongside on the large-N STL tasks to expose the conditioning failure the ridge fixes.

Experiments:
1. **Scaling / crossover** — throughput & peak memory vs N (FN overtakes full attention).
2. **Operator fidelity** — forward/backward relerr of scalar & tf32 pinv vs the fp32 reference (no training).
3. **Conditioning** — cond(K2) vs N and how the ridge bounds it (the math behind surpassing vanilla Nystromformer).
4. **CIFAR-10** (N=65, well-conditioned → kappa=0) and **STL-10** at 2304 / 9216 tokens (large N → kappa=1e3 ridge, plus the kappa=0 vanilla arm), all backends x seeds.
5. **MQAR** recall — short context (seq_len 256), so **kappa=0** (no ridge needed); all backends matched.

**GPU:** kernels are **sm_80+** — use **A100 or L4** (Colab Pro/Pro+). A **T4 will NOT work.**

## 0. Check the GPU

In [ ]:
import torch
name = torch.cuda.get_device_name(); cap = torch.cuda.get_device_capability()
print(name, '| compute capability', cap)
assert cap[0] >= 8, f'Need sm_80+ (A100/L4); this is {name} sm_{cap[0]}{cap[1]}. Switch runtime.'
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Get the code (clean clone + CUTLASS submodule)

In [ ]:
REPO_URL = 'https://github.com/athrva98/FlashNystrom.git'
REPO_DIR = '/content/flashnystrom'   # absolute repo root. Every cell below cd's here,
                                     # so a runtime restart or out-of-order run can't break paths.
# Fresh single clone (no nesting on re-run) WITH submodules
%cd /content
!rm -rf flashnystrom
!git clone --recurse-submodules $REPO_URL flashnystrom
%cd {REPO_DIR}
!cd "{REPO_DIR}" && git submodule update --init --recursive   # ensure CUTLASS (CuTe) headers
import os
assert os.path.isdir(f'{REPO_DIR}/third_party/cutlass/include'), \
    'CUTLASS submodule did not fetch -- re-run this cell (check network).'
print('CUTLASS headers present at', REPO_DIR, '- ok to build')

## 2. Build the fused CUDA kernels (~3-6 min)

In [ ]:
import os, torch
cap = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'
# Colab's gcc/CUDA toolchain trips -Werror on third-party headers; LAX disables that guard.
os.environ['FLASH_NYSTROM_LAX_BUILD'] = '1'
!cd "{REPO_DIR}" && pip install -e . --no-build-isolation

## 3. Verify the kernels

In [ ]:
import torch
from flash_nystrom import flash_nystrom_attention
from flash_nystrom.reference import nystrom_attention_reference
mk = lambda: torch.randn(4, 2, 256, 64, device='cuda', dtype=torch.bfloat16)
q, k, v = mk(), mk(), mk()
o = flash_nystrom_attention(q, k, v, 64, 6); r = nystrom_attention_reference(q, k, v, 64, 6)
print('fwd finite:', bool(torch.isfinite(o).all()), ' max|fn-ref|:', (o.float()-r.float()).abs().max().item())
qg = q.clone().requires_grad_(True); flash_nystrom_attention(qg, k, v, 64, 6).sum().backward()
print('bwd grad finite:', bool(torch.isfinite(qg.grad).all()))

## 4. Mount Drive + output folder
Re-run each session; keep `RUN_NAME` to accumulate, change it for a fresh run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RUN_NAME = 'flashnystrom_run'   # change for a separate run
OUTDIR = '/content/drive/MyDrive/flashnystrom_runs/' + RUN_NAME
RAW = OUTDIR + '/raw'           # per-(experiment, seed) raw results
SEEDS = [0, 1, 2]               # multi-seed
os.makedirs(OUTDIR + '/figures', exist_ok=True)
os.makedirs(RAW, exist_ok=True)
print('All results ->', OUTDIR)

# --- datasets: download ONCE ever, then restore from Drive on every session ---
# Tarballs persist in DATA_CACHE on Drive; each session copies them to fast
# local disk and torchvision's md5 check skips the download. CIFAR-10 uses the
# PyTorch CI S3 mirror (cs.toronto.edu throttles to ~10 kB/s).
import shutil, torchvision
DATA_CACHE = '/content/drive/MyDrive/flashnystrom_runs/data_cache'
DATA_DIR = '/content/data'
os.makedirs(DATA_CACHE, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.environ['FN_DATA_DIR'] = DATA_DIR   # train_three_way.py reads this
torchvision.datasets.CIFAR10.url = 'https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz'  # verified md5 c58f30108f718f92721af3b95e74349a
_TARBALLS = ['cifar-10-python.tar.gz', 'stl10_binary.tar.gz']
for f in _TARBALLS:                     # Drive cache -> local disk
    src_, dst_ = os.path.join(DATA_CACHE, f), os.path.join(DATA_DIR, f)
    if os.path.exists(src_) and not os.path.exists(dst_):
        print('restoring', f, 'from Drive cache'); shutil.copy(src_, dst_)
torchvision.datasets.CIFAR10(DATA_DIR, download=True)            # no-op if cached
torchvision.datasets.STL10(DATA_DIR, split='train', download=True)
for f in _TARBALLS:                     # local disk -> Drive cache (first time)
    src_, dst_ = os.path.join(DATA_DIR, f), os.path.join(DATA_CACHE, f)
    if os.path.exists(src_) and not os.path.exists(dst_):
        print('caching', f, 'to Drive'); shutil.copy(src_, dst_)
print('datasets ready in', DATA_DIR)


## 5. Scaling / crossover  -> `scaling.json` + log
Throughput + peak memory vs N at the auto-found max batch. FN overtakes full attention as N grows, and sdpa OOMs. (~10-20 min)

In [ ]:
cmd = ('python benchmarks/profile_scaling.py'
       ' --backends sdpa flash_nystrom flash_nystrom_tc nystrom_reference'
       ' --Ns 256 512 1024 2048 4096 8192 16384'
       f' --json {OUTDIR}/scaling.json 2>&1 | tee {OUTDIR}/scaling.log')
!cd "{REPO_DIR}" && {cmd}

## 5a. Operator fidelity -> `operator_fidelity.json`
The two pinv paths vs the fp32 reference (no training). The scalar path is faithful (forward ~3e-4; backward matches the reference autograd); the tf32 tensor-core path carries an N-independent ~1-3% forward error. This is the precision side of the precision/latency trade.

In [ ]:
# Operator fidelity (no training): scalar (default) and tf32-TC pinv vs the fp32 reference.
# Forward AND backward relerr. scalar ~ fp32-input floor (~3e-4); tf32-TC carries ~1-3% on the
# output (N-independent). Backward of the scalar path matches the reference autograd (faithful).
import torch, json
from flash_nystrom.flash_nystrom import flash_nystrom_attention
from flash_nystrom.reference import nystrom_attention_reference as nref
m, J, KAPPA = 64, 16, 1e3
relerr = lambda a, b: (a.float() - b.float()).norm().item() / b.float().norm().clamp_min(1e-12).item()
torch.manual_seed(0)
rows = []
for N in [256, 2304, 9216]:
    q = torch.randn(1, 4, N, 64, device='cuda'); k = torch.randn(1, 4, N, 64, device='cuda'); v = torch.randn(1, 4, N, 64, device='cuda')
    ref = nref(q.float(), k.float(), v.float(), m, J, None, 0, kappa_star=KAPPA)
    q16, k16, v16 = q.half(), k.half(), v.half()
    o_sc = flash_nystrom_attention(q16, k16, v16, num_landmarks=m, newton_iter=J, kappa_star=KAPPA, use_tc_pinv=False)
    o_tc = flash_nystrom_attention(q16, k16, v16, num_landmarks=m, newton_iter=J, kappa_star=KAPPA, use_tc_pinv=True)
    # backward: scalar grads vs reference autograd (same fp16 inputs, same upstream grad)
    g = torch.randn_like(q16)
    qa = q16.clone().requires_grad_(); ka = k16.clone().requires_grad_(); va = v16.clone().requires_grad_()
    flash_nystrom_attention(qa, ka, va, num_landmarks=m, newton_iter=J, kappa_star=KAPPA, use_tc_pinv=False).backward(g)
    qb = q16.clone().requires_grad_(); kb = k16.clone().requires_grad_(); vb = v16.clone().requires_grad_()
    nref(qb, kb, vb, m, J, None, 0, kappa_star=KAPPA).backward(g)
    r = {'N': N,
         'scalar_fwd_relerr': relerr(o_sc, ref), 'tc_fwd_relerr': relerr(o_tc, ref),
         'scalar_dQ_relerr': relerr(qa.grad, qb.grad), 'scalar_dV_relerr': relerr(va.grad, vb.grad)}
    rows.append(r); print(r)
json.dump(rows, open(f'{OUTDIR}/operator_fidelity.json', 'w'), indent=2)
print('wrote', OUTDIR + '/operator_fidelity.json')

## 5b. Conditioning -> `conditioning.json`
The landmark Gram K2 is ill-conditioned (cond ~1e5-1e7), so what vanilla Nystromformer's Newton-Schulz must invert, K2^T K2, has cond ~1e10-1e13 -- far past where NS converges. The Tikhonov ridge M = K2^T K2 + lambda*I caps cond(M) <= ~kappa_star (~1e3 here) regardless, which is what keeps the pinv convergent at large N. Training-free, fp64. (On trained models the gap widens further: segment-mean landmarks regress toward the global mean as N grows.)

In [ ]:
# Conditioning: cond(K2) grows with N; the Tikhonov ridge bounds what Newton-Schulz inverts.
# Vanilla Nystromformer (kappa=0) inverts K2^T K2 with cond ~ cond(K2)^2 (up to ~1e14 at 9216);
# the ridge M = K2^T K2 + lambda*I caps cond(M) <= ~kappa_star, keeping NS convergent. (fp64, no training)
import torch, json
m = 64
def landmark_K2(q, k):
    B, H, N, D = q.shape; s = D ** -0.25
    qs, ks = q * s, k * s; seg = N // m; tn = seg * (m - 1)
    qf = qs[:, :, :tn].reshape(B, H, m - 1, seg, D).mean(3)
    kf = ks[:, :, :tn].reshape(B, H, m - 1, seg, D).mean(3)
    ql = qs[:, :, tn:].mean(2, keepdim=True); kl = ks[:, :, tn:].mean(2, keepdim=True)
    qt = torch.cat([qf, ql], 2); kt = torch.cat([kf, kl], 2)
    return torch.softmax(qt @ kt.transpose(-2, -1), -1)   # (B,H,m,m)
def condmax(A):
    try: return torch.linalg.cond(A).flatten().max().item()
    except Exception: return float('inf')
torch.manual_seed(0)
rows = []
for N in [256, 1024, 4096, 9216]:
    q = torch.randn(1, 4, N, 64, device='cuda', dtype=torch.float64)
    k = torch.randn(1, 4, N, 64, device='cuda', dtype=torch.float64)
    K2 = landmark_K2(q, k)
    Kt = K2.transpose(-2, -1); M0 = Kt @ K2                       # what kappa=0 NS inverts
    n1 = K2.abs().sum(-2).amax(-1); ninf = K2.abs().sum(-1).amax(-1)
    lam = (n1 * ninf / 1e3)[..., None, None]
    eye = torch.eye(m, device='cuda', dtype=torch.float64)
    M1 = M0 + lam * eye                                           # kappa=1e3 ridge
    r = {'N': N, 'cond_K2': condmax(K2), 'cond_M_kappa0': condmax(M0), 'cond_M_kappa1e3': condmax(M1)}
    rows.append(r); print(r)
json.dump(rows, open(f'{OUTDIR}/conditioning.json', 'w'), indent=2)
print('wrote', OUTDIR + '/conditioning.json')

## 6. CIFAR-10 ViT (multi-seed) -> `raw/cifar_seed{S}.json`

In [ ]:
# CIFAR-10 (N=65): full grid: SDPA + {ref, FN, FN-TC} x {vanilla k=0, ridge k*=1000}. (resumable)
for s in SEEDS:
    out = f'{RAW}/cifar_full_seed{s}.json'
    if os.path.exists(out):
        print('skip', out); continue
    cmd = ('python benchmarks/train_three_way.py'
           ' --dataset cifar10 --patch_size 4 --epochs 20 --grad_clip 1.0 --kappa_star 1000'
           f' --seed {s} --backends sdpa nystrom_vanilla flash_nystrom_vanilla flash_nystrom_tc_vanilla nystrom_reference flash_nystrom flash_nystrom_tc'
           f' --out_json {out} 2>&1 | tee {RAW}/cifar_full_seed{s}.log')
    !cd "{REPO_DIR}" && {cmd}


## 7. STL-10 @ 2304 tokens (patch 2, multi-seed)

In [ ]:
# STL-10 @ 2304 tokens: full grid: SDPA + {ref, FN, FN-TC} x {vanilla, ridge k*=1000}. ~63 min/seed.
for s in SEEDS:
    out = f'{RAW}/stl10_p2_full_seed{s}.json'
    if os.path.exists(out):
        print('skip', out); continue
    cmd = ('python benchmarks/train_three_way.py'
           ' --dataset stl10 --patch_size 2 --epochs 50 --grad_clip 1.0 --kappa_star 1000'
           f' --seed {s} --backends sdpa nystrom_vanilla flash_nystrom_vanilla flash_nystrom_tc_vanilla nystrom_reference flash_nystrom flash_nystrom_tc'
           f' --out_json {out} 2>&1 | tee {RAW}/stl10_p2_full_seed{s}.log')
    !cd "{REPO_DIR}" && {cmd}
    if s == SEEDS[0]:
        !cp {out} {OUTDIR}/stl10_p2.json   # legacy path for the figures cell


## 7b. STL-10 @ 9216 tokens (patch 1, multi-seed, extreme N)

In [ ]:
# STL-10 @ 9216 tokens: full grid: {ref, FN, FN-TC} x {vanilla, ridge}. SDPA excluded (O(N^2)). ~2.5 h/seed.
for s in SEEDS:
    out = f'{RAW}/stl10_p1_full_seed{s}.json'
    if os.path.exists(out):
        print('skip', out); continue
    cmd = ('python benchmarks/train_three_way.py'
           ' --dataset stl10 --patch_size 1 --epochs 50 --autobatch --grad_clip 1.0 --kappa_star 1000'
           f' --seed {s} --backends nystrom_vanilla flash_nystrom_vanilla flash_nystrom_tc_vanilla nystrom_reference flash_nystrom flash_nystrom_tc'
           f' --out_json {out} 2>&1 | tee {RAW}/stl10_p1_full_seed{s}.log')
    !cd "{REPO_DIR}" && {cmd}
    if s == SEEDS[0]:
        !cp {out} {OUTDIR}/stl10_p1.json   # legacy path for the figures cell


### STL-10 @ 32K tokens: full grid (conditioning stress test)
STL upscaled to 180px, patch 1 -> N=32401 pixel tokens. cond(K2) grows with N; at 9216
tokens the ridge made no accuracy difference (and hurt at 2304). This probes whether a
no-ridge collapse appears at ~3.5x larger N, with the complete backend x ridge grid so
the paper table comes from one run. SDPA excluded (O(N^2) infeasible at 32K).


In [ ]:
# STL-10 @ 32401 tokens (180px, pixel patches): full 7-backend grid incl. the
# fp32 reference, {ref, ref-fp32, FN, FN-TC} x {vanilla k=0, ridge k=1000}, 3 seeds.
# SDPA excluded (O(N^2)). BATCH 12 (NOT 32, NOT autobatch): the fp32 reference arm
# trains in full fp32, which doubles the (N,m) softmax memory and OOMs at batch 32;
# batch 12 fits all seven arms (fp16 arms use far less). This is why fp32 never
# completed in the old batch-32 runs. --train_frac 0.5 = fixed 2500-image subset
# (identical across all arms within a seed). --kappa_star 1000 ridges the non-vanilla
# arms; *_vanilla arms are hardcoded k=0. NOTE: batch 12 makes every arm's numbers
# self-consistent but NOT comparable to any batch-32 run.
for s in SEEDS:
    out = f'{RAW}/stl10_32k_full_seed{s}.json'
    if os.path.exists(out):
        print('skip', out); continue
    cmd = ('python benchmarks/train_three_way.py'
           ' --dataset stl10 --img_size 180 --patch_size 1 --epochs 50'
           ' --batch_size 12 --train_frac 0.5 --grad_clip 1.0 --kappa_star 1000'
           f' --seed {s} --backends nystrom_vanilla flash_nystrom_vanilla flash_nystrom_tc_vanilla nystrom_reference nystrom_reference_fp32 flash_nystrom flash_nystrom_tc'
           f' --out_json {out} 2>&1 | tee {RAW}/stl10_32k_full_seed{s}.log')
    !cd "{REPO_DIR}" && {cmd}

## 8. MQAR recall (multi-seed) -> `raw/mqar_{backend}_seed{S}.json`
Zoology-faithful recipe (`paper/mqar/train.py`) with `--grad_clip 1.0`, all three backends x seeds.

In [ ]:
# MQAR: short context (seq_len 256, seg_len 4) -> K2 well-conditioned, so NO ridge (kappa=0).
# All backends matched at kappa=0 (the full-run collapse came from FN running kappa=1e3 while the
# reference ran kappa=0; the mqar_investigation notebook sweeps kappa* to confirm 0 is right here).
import re, json
for backend in ['sdpa', 'nystrom_reference', 'flash_nystrom', 'flash_nystrom_tc']:
    for s in SEEDS:
        out = f'{RAW}/mqar_{backend}_seed{s}.json'
        if os.path.exists(out):
            print('skip', out); continue
        log = f'{RAW}/mqar_{backend}_seed{s}.log'
        cmd = ('python -m paper.mqar.train'
               f' --backend {backend} --seed {s} --kappa_star 0'
               ' --seq_len 256 --num_kv_pairs 16 --num_landmarks 64'
               ' --newton_iter 6 --grad_clip 1.0 --autobatch'
               f' 2>&1 | tee {log}')
        !cd "{REPO_DIR}" && {cmd}
        txt = open(log).read()
        mm = re.search(r'best test recall:\s*([\d.]+)%', txt)
        json.dump({'experiment': 'mqar', 'backend': backend, 'seed': s, 'kappa_star': 0.0,
                   'recall': float(mm.group(1)) if mm else None},
                  open(out, 'w'), indent=2)
print('mqar done')

## 9. Aggregate seeds -> mean +/- std (`aggregated.json`)
Rolls every `raw/*.json` into the paper tables (vision test accuracy, MQAR recall).

In [ ]:
import glob, json, statistics as st
def agg(v):
    v = [x for x in v if x is not None]
    return (st.mean(v), st.pstdev(v) if len(v) > 1 else 0.0, len(v)) if v else (float('nan'), 0.0, 0)

vis = {}
for f in glob.glob(f'{RAW}/cifar*_seed*.json') + glob.glob(f'{RAW}/stl10_*_seed*.json'):
    d = json.load(open(f)); exp = os.path.basename(f).split('_seed')[0]
    for r in d['results']:
        vis.setdefault((exp, r['label']), []).append(r['test_acc'])
print('VISION (test acc %, mean +/- std):')
for (e, b), a in sorted(vis.items()):
    mu, sd, n = agg(a); print(f'  {e:10s} {b:14s}: {mu:5.1f} +/- {sd:.1f} (n={n})')

mq = {}
for f in glob.glob(f'{RAW}/mqar_*_seed*.json'):
    d = json.load(open(f)); mq.setdefault(d['backend'], []).append(d.get('recall'))
print('MQAR (recall %, mean +/- std):')
for b, v in sorted(mq.items()):
    mu, sd, n = agg(v); print(f'  {b:18s}: {mu:5.2f} +/- {sd:.2f} (n={n})')

json.dump({'vision': {f'{e}|{b}': agg(a) for (e, b), a in vis.items()},
           'mqar':   {b: agg(v) for b, v in mq.items()}},
          open(f'{OUTDIR}/aggregated.json', 'w'), indent=2)
print('wrote', OUTDIR + '/aggregated.json')

## 10. Build figures (PDF + PNG) from the saved JSON
Re-run anytime to re-style — reads only the JSON, no GPU.

In [ ]:
cmd = ('python benchmarks/make_figures.py'
       f' --scaling {OUTDIR}/scaling.json'
       f' --cifar {OUTDIR}/stl10_p2.json'
       f' --outdir {OUTDIR}/figures')
!cd "{REPO_DIR}" && {cmd}

## 11. Inventory + zip (already on Drive)

In [ ]:
import glob, zipfile
arts = sorted(glob.glob(OUTDIR+'/figures/*') + glob.glob(OUTDIR+'/*.json') + glob.glob(OUTDIR+'/*.log'))
print('Saved to Drive:', OUTDIR)
for p in arts: print('  ' + os.path.relpath(p, OUTDIR))
zf = OUTDIR + '/artifacts.zip'
with zipfile.ZipFile(zf, 'w') as z:
    for p in arts: z.write(p, os.path.relpath(p, OUTDIR))
print('zipped ->', zf)

## Notes
- Nothing is lost on disconnect — everything is in `OUTDIR` on Drive.
- Re-style plots with no GPU: re-run section 8.
- STL-10 absolute accuracy is modest (from-scratch ViT on 5k images); the claim is **FN matches full attention + scales better**, not SOTA.
- Preview a figure: `from IPython.display import Image; Image(OUTDIR+'/figures/scaling.png')`.